In [ ]:
from torch import nn
import torch

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

saving incolab

In [ ]:
base_path = '/content/drive/My Drive/esrgan'
import os
# os.mkdir(base_path)
# import os

# Path to your Google Drive folder
base_path = '/content/drive/My Drive/esrgan'

# Create main folder
os.makedirs(base_path, exist_ok=True)

# Create subfolders
os.makedirs(os.path.join(base_path, 'saved'), exist_ok=True)
os.makedirs(os.path.join(base_path, 'test_images'), exist_ok=True)

# File names
files = ['ESRGAN.png', 'config.py', 'dataset.py', 'loss.py', 'model.py', 'train.py', 'utils.py']

# Create files in the main folder
for file_name in files:
    file_path = os.path.join(base_path, file_name)
    if not os.path.exists(file_path):  # Avoid overwriting existing files
        with open(file_path, 'w') as file:
            pass  # Create empty file


In [ ]:
# %%writefile "/content/drive/MyDrive/esrgan/model.py"
class ConvBlock(nn.Module):
  def __init__(self,in_channels,out_channels,use_act,**kwargs):
    super().__init__()
    self.cnn=nn.Conv2d(
        in_channels,
        out_channels,
        **kwargs,
        bias=True
    )
    self.act=nn.LeakyReLU(0.2,inplace=True) if use_act else nn.Identity()

  def forward(self,x):
      return self.act(self.cnn(x))


class UpsampleBlock(nn.Module):
  def __init__(self,in_c,scale_factor=2):
    super().__init__()
    self.upsample=nn.Upsample(scale_factor=scale_factor,mode='nearest')
    self.conv=nn.Conv2d(in_c,in_c,3,1,1,bias=True)
    self.act=nn.LeakyReLU(0.2,inplace=True)
  def forward(self,x):
    return self.act(self.conv(self.upsample(x)))

class DenseResidualBlock(nn.Module):
  def __init__(self,in_channels,channels=32,residual_beta=0.2):
    super().__init__()
    self.residual_beta=residual_beta
    self.blocks=nn.ModuleList()  #for maintaining paramters when u have to use loops in forward function
    for i in range(5):
      self.blocks.append(
          ConvBlock(
              in_channels+channels*i,
              channels if i<=3 else in_channels,
              kernel_size=3,
              stride=1,
              padding=1,
              use_act=True if i<=3 else False,
          )
      )
  def forward(self,x):
      new_inputs=x
      for block in self.blocks:
         out=block(new_inputs)
         new_inputs=torch.concat([new_inputs,out],dim=1)

      return self.residual_beta*out+x

class RRDB(nn.Module):
  def __init__(self,in_channels,residual_beta=0.2): #self.rrdb = nn.Sequential(*[DenseResidualBlock(in_channels) for _ in range(3)])
    super().__init__()
    self.residual_beta=residual_beta
    self.rrdb=nn.Sequential(*[DenseResidualBlock(in_channels) for _ in range(3)])
  def forward(self,x):
       return self.rrdb(x)* self.residual_beta+x

class Generator(nn.Module):
  def __init__(self,in_channels=3,num_channels=64,num_blocks=23):
     super().__init__()
     self.initial=nn.Conv2d(
      in_channels,
      num_channels,
      kernel_size=3,
      stride=1,
      padding=1,
      bias=True
  )
     self.residuals=nn.Sequential(*[RRDB(num_channels)for _ in range(num_blocks)])
     self.conv=nn.Conv2d(num_channels,num_channels,kernel_size=3,stride=1,padding=1)
     self.upsample=nn.Sequential(
         UpsampleBlock(num_channels),
         UpsampleBlock(num_channels),
     )
     self.final=nn.Sequential(
         nn.Conv2d(num_channels,num_channels,3,1,1,bias=True),
         nn.LeakyReLU(0.2,inplace=True),
         nn.Conv2d(num_channels,in_channels,3,1,1,bias=True)
     )
  def forward(self,x):
        initial=self.initial(x)
        x=self.conv(self.residuals(initial))+initial
        x=self.upsample(x)

        return self.final(x)


class Discriminator(nn.Module):
  def __init__(self,in_channels=3,features=[64,64,128,128,256,256,512,512]):
    super().__init__()
    blocks=[]
    for idx,feature in enumerate(features):
      blocks.append(
          ConvBlock(
              in_channels,
              feature,
              kernel_size=3,

              stride=1 if idx % 2 == 0 else 2,
              padding=1,
              use_act=True
          ),

      )
      in_channels=feature
    self.blocks=nn.Sequential(*blocks)
    self.classifier=nn.Sequential(
        nn.AdaptiveAvgPool2d((6,6)),
        nn.Flatten(),
        nn.Linear(features[-1]*6*6 ,1024),
        nn.LeakyReLU(0.2,inplace=True),
        nn.Linear(1024,1)
    )
  def forward(self,x):
      x=self.blocks(x)
      return self.classifier(x)
def initializa_weight(model,scale=0.1):
  for m in model.modules():
    if isinstance(m,nn.Conv2d):
      nn.init.kaiming_normal_(m.weight.data)
      m.weight.data*=scale
    elif ininstance(m,nn.Linear):
      nn.init.kaiming_normal_(m.weight.data)
      m.weight.data*=scale
def test():
    low_resolution = 24  # 96x96 -> 24x24
    with torch.cuda.amp.autocast():
        x = torch.randn((5, 3, low_resolution, low_resolution))
        gen = Generator()
        gen_out = gen(x)
        disc = Discriminator()
        disc_out = disc(gen_out)

        print(gen_out.shape)
        print(disc_out.shape)


if __name__ == "__main__":
    test()






<ipython-input-28-c5f4226402db>:127: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/usr/local/lib/python3.11/dist-packages/torch/amp/autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


torch.Size([5, 3, 96, 96])
torch.Size([5, 1])


loss.py

In [ ]:
# !pip install torchvision

In [ ]:
# !pip install config

In [ ]:
# %%writefile "/content/drive/MyDrive/esrgan/loss.py"
# import torch.nn as nn
# from torchvision.models import vgg19
# import config

# class VGGLoss(nn.Module):
#   def __init__(self):
#     super().__init__()
#     self.vgg=vgg19(pretrained=True).features[:35].eval().to(config.DEVICE)

#     for param in self.vgg.paramters():
#       param.requires_grad=False

#     self.loss=nn.MSELoss()
#   def forward(self,input,target):
#     vgg_input_features=self.vgg(input)
#     vgg_target_features=self.vgg(target)
#     return self.loss(vgg_input_features,vgg_target_features)


Overwriting /content/drive/MyDrive/esrgan/loss.py
Overwriting /content/drive/MyDrive/esrgan/loss.py


In [ ]:
%%writefile "/content/drive/MyDrive/esrgan/loss.py"

import torch
import torch.nn as nn
from torchvision.models import vgg19
import config

class VGGLoss(nn.Module):
    def __init__(self):
        super(VGGLoss, self).__init__()
        # Load the VGG19 model and extract features up to layer 35
        self.vgg = vgg19(pretrained=True).features[:35].eval().to(config.DEVICE)

        # Freeze the parameters of the VGG model
        for param in self.vgg.parameters():
            param.requires_grad = False

        # Use Mean Squared Error loss
        self.loss = nn.MSELoss()

    def forward(self, input, target):
        # Extract features using the VGG19 model
        vgg_input_features = self.vgg(input)
        vgg_target_features = self.vgg(target)
        # Calculate the MSE loss between the features
        return self.loss(vgg_input_features, vgg_target_features)

Overwriting /content/drive/MyDrive/esrgan/loss.py


In [ ]:
note='AI_GENEERATED Dataset.py'
import torch
import os
from torch.utils.data import Dataset, DataLoader
import config
import cv2

class MyImageFolder(Dataset):
    def __init__(self, root_dir):
        super(MyImageFolder, self).__init__()
        self.data = []
        self.root_dir = root_dir
        self.class_names = os.listdir(root_dir)

        for index, name in enumerate(self.class_names):
            files = os.listdir(os.path.join(root_dir, name))
            self.data += list(zip(files, [index] * len(files)))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        img_file, label = self.data[index]
        root_and_dir = os.path.join(self.root_dir, self.class_names[label])

        # Load and process the image
        image_path = os.path.join(root_and_dir, img_file)
        image = cv2.imread(image_path)
        if image is None:
            raise ValueError(f"Failed to load image at {image_path}")

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        both_transform = config.both_transforms(image=image)['image']
        low_res = config.lowres_transform(image=both_transform)['image']
        high_res = config.highres_transform(image=both_transform)['image']

        return low_res, high_res

# Test Function
def test():
    root_dir = '/content/drive/My Drive/esrgan/test_images'
    if not os.path.exists(root_dir):
        raise ValueError(f"Root directory '{root_dir}' does not exist!")

    dataset = MyImageFolder(root_dir=root_dir)
    loader = DataLoader(dataset, batch_size=8)

    for i, (low_res, high_res) in enumerate(loader):
        print(f"Batch {i + 1}:")
        print("Low-res shape:", low_res.shape)
        print("High-res shape:", high_res.shape)

# Main Script
if __name__ == '__main__':
    test()


dataset.py

In [ ]:
%%writefile "/content/drive/MyDrive/esrgan/dataset.py"

import torch
import tqdm as tqdm
import time
import torch.nn
import os
from torch.utils.data import Dataset,DataLoader
import numpy as np
import config
from PIL import Image
import cv2

class MyImageFolder(Dataset):
  def __init__(self,root_dir):
    super(MyImageFolder,self).__init__()
    self.data=[]
    self.root_dir=root_dir
    self.class_names=os.listdir(root_dir)

    for index,name in enumerate(self.class_names):
      files=os.listdir(os.path.join(root_dir,name))
      self.data+=list(zip(files,[index]*len(files)))
  def __len__(self):
    return len(self.data)
  def __getitem__(self,index):
    img_file,label=self.data[index]
    root_and_dir=os.path.join(self.root_dir,self.class_names[label])

    image=cv2.imread(os.path.join(root_and_dir,img_file)) #picking one by one image
    image=cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
    both_transform=config.both_transforms(image=image)['image']
    low_res=config.lowres_transform(image=both_transform)['image']
    high_res=config.highres_transform(image=both_transform)['image']
    return low_res,high_res

  def test():
    root_dir = '/content/drive/My Drive/esrgan/test_images'
    dataset=MyImageFolder(root_dir=root_dir)
    loader=DataLoader(dataset,batch_size=8)

    for low_res,high_res in loader:
      print(low_res.shape)
      print(high_res.shape)
if __name__=='__main__':
    test()



Overwriting /content/drive/MyDrive/esrgan/dataset.py


config.py

In [ ]:
%%writefile "/content/drive/MyDrive/esrgan/config.py"
import torch
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2

LOAD_MODEL=True
SAVE_MODEL=True
CHECKPOINT_GEN='gen_pth'
CHECKPOINT_DISC='disc_pth'
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
LEARNING_RATE=1e-4
NUM_EPOCHS=10000
BATCH_SIZE=16
LAMBDA_GP=10
NUM_WORKERS=4
HIGH_RES=128
LOW_RES=HIGH_RES//4
IMG_CHANNELS=3

highres_transform=A.Compose(
    [
        A.Normalize(mean=[0,0,0], std=[1, 1, 1]),
        ToTensorV2()
    ]
)

lowres_transform=A.Compose(
    [
        A.Resize(width=LOW_RES, height=LOW_RES, interpolation=Image.BICUBIC),
        A.Normalize(mean=[0 ,0, 0],std=[1, 1, 1]),
        ToTensorV2()

    ]
)

both_transforms=A.Compose(
    [
        A.RandomCrop(width=HIGH_RES,height=HIGH_RES),
        A.HorizontalFlip(p=0.5),
        A.RandomRotate90(p=0.5)
    ]
)

test_transform=A.Compose(
    [
        A.Normalize(means=[0, 0, 0], std=[1, 1, 1,]),
        ToTensorV2()
    ]
)

Overwriting /content/drive/MyDrive/esrgan/config.py


utils.py

In [ ]:
%%writefile "/content/drive/MyDrive/esrgan/utils.py"
import torch
import os
import config
import numpy as np
from PIL import Image
from torchvision.utils import save_image


def gradient_penalty(critic,real,fake,device):
  BATCH_SIZE,C,H,W=real.shape
  alpha=torch.rand(BATCH_SIZE,1,1,1).repeat(1,C,H,W).to(device)
  interpolated_images=real*alpha+fake.detach()*(1-alpha)
  interpolated_images.requires_grad_(True)

  #calculate critic scores
  mixed_scores=critic(interpolated_images)
  #take gradient of the score with respect to the images
  gradient=torch.autograd.grad(
      inputs=interpolated_images,
      outputs=mixed_scores,
      grad_outputs=torch.ones_like(mixed_scores),
      create_graph=True,
      retain_graph=True)[0]

  gradient=gradient.view(gradient.shape[0],-1)
  gradient_norm=gradient.norm(2,dim=1)
  gradient_penalty=torch.mean((gradient_norm-1)**2)
  return gradient_penalty

def save_checkpoint(model,optimizer,filename='my_checkpoint.pth.tar'):
  print('=>Saving checkpoint')
  checkpoint={
      'state_dict':model.state_dict(),
      'optimizer':optimizer.state_dict()
  }
  torch.save(checkpoint,filename)

def load_checkpoint(checkpoint_file,model,optimizer,lr):
  print('=>loading checkpoint')
  checkpoint=torch.load(checkpoint_file,map_location=config.DEVICE)
  #model.load_state_dict(checkpoint)
  model.load_state_dict(checkpoint['state_dict'])
  optimizer.load_state_dict(checkpoint['optimizer'])

  #if we dont do this then it will just have learning rate of old checkpoint
  #and it will lead to many hours of debugging

  for param_group in optimizer.param_groups:
    param_group['lr']=lr

def plot_examples(low_res_folder,gen):
  files=os.listdir(low_res_folder)

  gen.eval()
  for file in files:
    image=Image.open('/content/drive/MyDrive/esrgan/test_images'+file)
    with torch.no_grad():
      upscaled_img=gen(
          config.test_transform(image=np.asarray(image))['image'].unsqueeze(0)
          .to(config.DEVICE)
      )
      save_image(upscaled_img,f'Saved/{file}')
    gen.train()

Overwriting /content/drive/MyDrive/esrgan/utils.py


In [ ]:
!pip install utils

  Preparing metadata (setup.py) ... done
  Created wheel for utils: filename=utils-1.0.2-py2.py3-none-any.whl size=13906 sha256=75807dc562377c6c937fced81d860e6fbf28251e062d9eea09f3e4653285c57b
  Stored in directory: /root/.cache/pip/wheels/15/0c/b3/674aea8c5d91c642c817d4d630bd58faa316724b136844094d
Successfully built utils


In [ ]:
import utils

train.py

In [ ]:
!ls /content/drive/MyDrive/esrgan


config.py   ESRGAN.png	model.py  test_images  utils.py
dataset.py  loss.py	saved	  train.py


In [ ]:
import torch.nn as nn

In [ ]:
#  %%writefile '/content/drive/MyDrive/esrgan/train.py'

import os
os.chdir('/content/drive/MyDrive/esrgan')  # Changed to the directory containing method.py
import sys
sys.path.append('/content/drive/MyDrive/esrgan')  # Changed to the directory containing method.py
import torch
import config
from torch import nn
from torch import optim
from utils import gradient_penalty, load_checkpoint, save_checkpoint, plot_examples
from loss import VGGLoss
from torch.utils.data import DataLoader
from model import Generator, Discriminator  # initialize_weights
from tqdm import tqdm
from dataset import MyImageFolder
from torch.utils.tensorboard import SummaryWriter

torch.backends.cudnn.benchmark = True

def train_fn(
    loader,
    disc,
    gen,
    opt_gen,
    opt_disc,
    l1,
    vgg_loss,
    g_scaler,
    d_scaler,
    writer,
    tb_step,
):
    loop = tqdm(loader, leave=True)

    for idx, (low_res, high_res) in enumerate(loop):
        high_res = high_res.to(config.DEVICE)
        low_res = low_res.to(config.DEVICE)

        with torch.cuda.amp.autocast():
            fake = gen(low_res)
            critic_real = disc(high_res)
            critic_fake = disc(fake.detach())
            gp = gradient_penalty(disc, high_res, fake, device=config.DEVICE)
            loss_critic = (
                -(torch.mean(critic_real) - torch.mean(critic_fake))
                + config.LAMBDA_GP * gp
            )

        opt_disc.zero_grad()
        d_scaler.scale(loss_critic).backward()
        d_scaler.step(opt_disc)
        d_scaler.update()

        # Train Generator: min log(1 - D(G(z))) <-> max log(D(G(z)))
        with torch.cuda.amp.autocast():
            l1_loss = 1e-2 * l1(fake, high_res)
            adversarial_loss = 5e-3 * -torch.mean(disc(fake))
            loss_for_vgg = vgg_loss(fake, high_res)
            gen_loss = l1_loss + loss_for_vgg + adversarial_loss

        opt_gen.zero_grad()
        g_scaler.scale(gen_loss).backward()
        g_scaler.step(opt_gen)
        g_scaler.update()

        writer.add_scalar("Critic loss", loss_critic.item(), global_step=tb_step)
        tb_step += 1

        if idx % 100 == 0 and idx > 0:
            plot_examples("/content/drive/MyDrive/esrgan/test_images", gen)

        loop.set_postfix(
            gp=gp.item(),
            critic=loss_critic.item(),
            l1=l1_loss.item(),
            vgg=loss_for_vgg.item(),
            adversarial=adversarial_loss.item(),
        )

    return tb_step


def main():
    # dataset = MyImageFolder(root_dir="/content/drive/MyDrive/esrgan/test_images")
    # loader = DataLoader(
    #     dataset,
    #     batch_size=config.BATCH_SIZE,
    #     shuffle=True,
    #     pin_memory=True,
    #     num_workers=config.NUM_WORKERS,
    # )

    gen = Generator(in_channels=3).to(config.DEVICE)
    disc = Discriminator(in_channels=3).to(config.DEVICE)
    initialize_weights(gen)

    opt_gen = optim.Adam(gen.parameters(), lr=config.LEARNING_RATE, betas=(0.0, 0.9))
    opt_disc = optim.Adam(disc.parameters(), lr=config.LEARNING_RATE, betas=(0.0, 0.9))
    writer = SummaryWriter("logs")
    tb_step = 0
    l1 = nn.L1Loss()
    gen.train()
    disc.train()
    vgg_loss = VGGLoss()

    g_scaler = torch.cuda.amp.GradScaler()
    d_scaler = torch.cuda.amp.GradScaler()

    if config.LOAD_MODEL:
        load_checkpoint(
            config.CHECKPOINT_GEN,
            gen,
            opt_gen,
            config.LEARNING_RATE,
        )
        # load_checkpoint(
        #     config.CHECKPOINT_DISC,
        #     disc,
        #     opt_disc,
        #     config.LEARNING_RATE,
        # )

    # You should not call sys.exit() here. It prematurely ends the script.
    # This line was removed to allow training to run
    for epoch in range(config.NUM_EPOCHS):
        tb_step = train_fn(
            loader,
            disc,
            gen,
            opt_gen,
            opt_disc,
            l1,
            vgg_loss,
            g_scaler,
            d_scaler,
            writer,
            tb_step,
        )

        if config.SAVE_MODEL:
            save_checkpoint(gen, opt_gen, filename=config.CHECKPOINT_GEN)
            save_checkpoint(disc, opt_disc, filename=config.CHECKPOINT_DISC)


if __name__ == "__main__":
    try_model = True

    if try_model:
        # Will just use pretrained weights and run on images
        # in test_images/ and save the ones to SR in saved/
        gen = Generator(in_channels=3).to(config.DEVICE)
        opt_gen = optim.Adam(gen.parameters(), lr=config.LEARNING_RATE, betas=(0.0, 0.9))
        # load_checkpoint(
        #     config.CHECKPOINT_GEN,
        #     gen,
        #     opt_gen,
        #     config.LEARNING_RATE,
        # )
        plot_examples("/content/drive/MyDrive/esrgan/test_images", gen)
    else:
        # This will train from scratch
        main()



FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/esrgan/test_imagesbaboon_LR.png'